## Training a small transformer to solve addition

### Data

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import torch
import torch.nn as nn
import torch.nn.functional as torch_f

import json
import random
import time
from pathlib import Path

In [2]:
class TrainConfig:
    def __init__(self):
        self.__dict__.update(
            steps=10000, batch_size=256,
            lr=2e-4, weight_decay=1e-2, grad_clip=1.0, eval_every=1000,
            eval_samples=500, min_digits=1, max_digits=7, max_seq_len=50,
            d_model=256, n_heads=8, n_layers=6,
        )

    def to_dict(self):
        return dict(vars(self))

cfg = TrainConfig()

In [3]:
Path("small_llm_training_artifacts").mkdir(parents=True, exist_ok=True)

In [4]:
vocab = list('0123456789+=;_')
pad_token = '_'
pad_id = vocab.index(pad_token)
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = dict(enumerate(vocab))

def encode(text):
    return [stoi[ch] for ch in text]

def decode(token_ids):
    chars = [itos[idx] for idx in token_ids]
    return ''.join(ch for ch in chars if ch != pad_token)

def sample_number(num_digits, rng):
    if num_digits == 1:
        return rng.randint(0, 9)
    return rng.randint(10 ** (num_digits - 1), 10 ** num_digits - 1)

def make_example(rng, min_digits, max_digits):
    a = sample_number(rng.randint(min_digits, max_digits), rng)
    b = sample_number(rng.randint(min_digits, max_digits), rng)
    return f'{a}+{b}={a+b};', a, b

def make_batch(rng, batch_size, device, min_digits, max_digits):
    seqs = [encode(make_example(rng, min_digits, max_digits)[0]) for _ in range(batch_size)]
    max_len = max(len(seq) for seq in seqs) - 1
    x = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    y = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    for i, seq in enumerate(seqs):
        x[i, :len(seq) - 1] = torch.tensor(seq[:-1], dtype=torch.long)
        y[i, :len(seq) - 1] = torch.tensor(seq[1:], dtype=torch.long)
    return x.to(device), y.to(device)

def has_carry(a, b):
    carry = 0
    while a > 0 or b > 0:
        total = a % 10 + b % 10 + carry
        if total >= 10:
            return True
        carry = total // 10
        a //= 10
        b //= 10
    return False

### Simple GPT implementation

In [5]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads, mlp_mult, dropout):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_mult * d_model), nn.GELU(),
            nn.Linear(mlp_mult * d_model, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask):
        h = self.ln_1(x)
        h, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        return x + h + self.mlp(self.ln_2(x + h))

class GPTConfig:
    def __init__(self, vocab_size=None, max_seq_len=64, d_model=256, n_heads=8, n_layers=6, mlp_mult=4, dropout=0.0):
        self.__dict__.update(
            vocab_size=len(vocab) if vocab_size is None else vocab_size,
            max_seq_len=max_seq_len, d_model=d_model, n_heads=n_heads,
            n_layers=n_layers, mlp_mult=mlp_mult, dropout=dropout,
        )

    def to_dict(self):
        return dict(vars(self))

class LLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg.d_model, cfg.n_heads, cfg.mlp_mult, cfg.dropout) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, idx):
        bsz, seq_len = idx.shape
        pos = torch.arange(seq_len, device=idx.device).unsqueeze(0).expand(bsz, seq_len)
        mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=idx.device), diagonal=1)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x, mask)
        return self.head(self.ln_f(x))

### Training

In [6]:
device = "cuda"
torch.manual_seed(1)
random.seed(1)

model_cfg = GPTConfig(max_seq_len=cfg.max_seq_len, d_model=cfg.d_model, n_heads=cfg.n_heads, n_layers=cfg.n_layers)
model = LLM(model_cfg).to(device)

sum(p.numel() for p in model.parameters())

4759040

In [7]:
@torch.no_grad()
def greedy_answer(model, prompt, device, max_new_tokens=10):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        if idx.shape[1] >= model.cfg.max_seq_len:
            break
        nxt = torch.argmax(model(idx)[:, -1, :], dim=-1, keepdim=True)
        idx = torch.cat([idx, nxt], dim=1)
        if int(nxt.item()) == stoi[';']:
            break
    return decode(idx[0].tolist()[len(prompt):]).split(';', 1)[0]

@torch.no_grad()
def evaluate_exact_match(model, device, n_samples, min_digits, max_digits, seed):
    rng = random.Random(seed)
    stats = {'all': [0, 0], 'carry': [0, 0], 'no_carry': [0, 0]}
    model.eval()

    for _ in range(n_samples):
        _, a, b = make_example(rng, min_digits, max_digits)
        ok = int(greedy_answer(model, f'{a}+{b}=', device, max_digits + 2) == str(a + b))
        group = 'carry' if has_carry(a, b) else 'no_carry'
        for name in ['all', group]:
            stats[name][0] += ok
            stats[name][1] += 1

    return {
        'exact_match': stats['all'][0] / max(stats['all'][1], 1),
        'exact_match_carry': stats['carry'][0] / max(stats['carry'][1], 1),
        'exact_match_no_carry': stats['no_carry'][0] / max(stats['no_carry'][1], 1),
        'n_total': float(stats['all'][1]),
        'n_carry': float(stats['carry'][1]),
        'n_no_carry': float(stats['no_carry'][1]),
    }



optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay, betas=(0.9, 0.95))
rng_train = random.Random(200000)
history = []

for step in range(cfg.steps + 1):
    model.train()
    x, y = make_batch(rng_train, cfg.batch_size, device, cfg.min_digits, cfg.max_digits)
    loss = torch_f.cross_entropy(model(x).view(-1, model_cfg.vocab_size), y.view(-1), ignore_index=pad_id)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    optimizer.step()

    if step % cfg.eval_every == 0:
        metrics = evaluate_exact_match(model, device, cfg.eval_samples, cfg.min_digits, cfg.max_digits, 1 + step)
        row = {'step': step, 'loss': loss.item(), **metrics}
        history.append(row)
        print(f"step={step} | loss={row['loss']:.2f} | carry={row['exact_match_carry']:.2f} | no_carry={row['exact_match_no_carry']:.2f}")

step=0 | loss=2.77 | carry=0.00 | no_carry=0.00
step=1000 | loss=1.67 | carry=0.12 | no_carry=0.51
step=2000 | loss=1.48 | carry=0.44 | no_carry=0.81
step=3000 | loss=1.42 | carry=0.49 | no_carry=0.82
step=4000 | loss=1.38 | carry=0.57 | no_carry=0.93
step=5000 | loss=1.35 | carry=0.83 | no_carry=0.98
step=6000 | loss=1.34 | carry=0.87 | no_carry=0.98
step=7000 | loss=1.32 | carry=0.92 | no_carry=0.99
step=8000 | loss=1.32 | carry=0.95 | no_carry=0.99
step=9000 | loss=1.32 | carry=0.95 | no_carry=0.99
step=10000 | loss=1.32 | carry=0.96 | no_carry=1.00


In [8]:
final_metrics = evaluate_exact_match(model, device, 2000, cfg.min_digits, cfg.max_digits, 1 + 10000)
out_dir = Path("small_llm_training_artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = out_dir / 'llm.pt'

In [9]:
summary = {'config': model_cfg.to_dict(), 'final_metrics': final_metrics, 'checkpoint': str(ckpt_path)}
torch.save({'model_state': model.state_dict(), 'args': cfg.to_dict(), **summary}, ckpt_path)
print(json.dumps(summary, indent=4))

{
    "config": {
        "vocab_size": 14,
        "max_seq_len": 50,
        "d_model": 256,
        "n_heads": 8,
        "n_layers": 6,
        "mlp_mult": 4,
        "dropout": 0.0
    },
    "final_metrics": {
        "exact_match": 0.972,
        "exact_match_carry": 0.9628378378378378,
        "exact_match_no_carry": 0.9980769230769231,
        "n_total": 2000.0,
        "n_carry": 1480.0,
        "n_no_carry": 520.0
    },
    "checkpoint": "small_llm_training_artifacts/llm.pt"
}


In [10]:
payload = torch.load(str(ckpt_path), map_location='cpu')
model = LLM(GPTConfig(**payload['config']))
model.load_state_dict(payload['model_state'])
model = model.to(device).eval()

for a, b in [(12, 34), (11199, 1), (99, 1), (999, 1), (99999, 1)]:
    prompt = f'{a}+{b}='
    pred = greedy_answer(model, prompt, device, max_new_tokens=15)
    print(f'{prompt} pred={pred} gt={a+b}')

12+34= pred=46 gt=46
11199+1= pred=11200 gt=11200
99+1= pred=100 gt=100
999+1= pred=1000 gt=1000
99999+1= pred=100000 gt=100000
